In [3]:
#imports
import os, glob, json
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as sps

from enterprise.pulsar import Pulsar
import enterprise.signals.parameter as parameter
from enterprise.signals import utils
from enterprise.signals import signal_base
from enterprise.signals import selections
from enterprise.signals import white_signals
from enterprise.signals import gp_signals
from enterprise.signals import selections
from enterprise.signals.selections import Selection

from enterprise_extensions import models, hypermodel
from enterprise_extensions.model_utils import bayes_fac
from la_forge import core as co
from la_forge import diagnostics as dg

from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc

import pickle

Optional mpi4py package is not installed.  MPI support is not available.


# Get data

In [7]:
pkl_file = '/home/mitch/test_quick_burst/15_year_data/psrs_trimmed_SNR99p.pkl'
with open(pkl_file, 'rb') as file:
    psrs = pickle.load(file)

# Define my model

In [ ]:
psr = psrs[0]
tmax = psr.toas.max()
tmin = psr.toas.min()
tspan = (tmax-tmin) / (365*25*3600)
bins = 30
freq = np.arange(tspan/bins, (bins+0.001)*tspan/bins, tspan/bins)



[ 0.49906384  0.99812767  1.49719151  1.99625535  2.49531919  2.99438302
  3.49344686  3.9925107   4.49157453  4.99063837  5.48970221  5.98876604
  6.48782988  6.98689372  7.48595756  7.98502139  8.48408523  8.98314907
  9.4822129   9.98127674 10.48034058 10.97940442 11.47846825 11.97753209
 12.47659593 12.97565976 13.4747236  13.97378744 14.47285127 14.97191511]


In [20]:


selection = selections.Selection(selections.by_backend)
selection_ng = selections.Selection(selections.nanograv_backends)

#WN
efac = parameter.Uniform(0.01,10.0)
equad = parameter.Uniform(-8.5,-5)
ecorr = parameter.Uniform(-8.5,-5)

wn = white_signals.MeasurementNoise(efac=efac, log10_t2equad=equad)
ec = white_signals.EcorrKernelNoise(log10_ecorr=ecorr, selection=selection_ng)

#timing model
tm = gp_signals.TimingModel(use_svd=True)

#RN

log10_A = parameter.Uniform(-20,-11)
gamma = parameter.Uniform(0,7)

pl = utils.powerlaw(log10_A=log10_A, gamma=gamma)
rn = gp_signals.FourierBasisGP(spectrum=pl, modes=freq)

#full model
s = wn + ec + tm + rn

pta = signal_base.PTA(s(psr))

In [23]:
print(pta.get_lnlikelihood)

<bound method PTA.get_lnlikelihood of <Enterprise PTA object: B1855+09>>
